# Black-box memory eval, step by step (dummy data)

**Given only what a production agent logs** (each call's `query`, `context` and `response`, plus the
user's other messages), find memory failures, say *where* they happened, and score them with denominators.
No ground truth goes into steps 1 to 7; step 8 reveals it to check the verdicts.

All data here is synthetic and tiny (1 user, 4 calls). It shows the method, not a benchmark result.
The LLM is a local open-source model (Qwen2.5 via Ollama). Each call to it is tagged with its role.

| Step | What | Needs history? | Uses LLM? |
|---|---|---|---|
| 0 | Raw agent logs | | |
| 1 | Normalize into call records | | |
| 2 | Per-call symptom checks (rules) | no | no |
| 3 | Extract structured facts from user messages, with quotes | yes | `extractor` |
| 4 | Grounding check: drop facts whose quote isn't in the message | yes | no |
| 5 | Resolve each fact's latest value (the reference) | yes | no |
| 6 | Check each call's context: relevance and conflicts | no | `judge_relevance`, `extractor` |
| 7 | Judge sufficiency and use; locate the failure | yes | `judge_sufficiency`, `judge_use` |
| 8 | Compare with hidden gold labels | | |
| 9 | Scores with denominators | | |

In [1]:
import json
import os
import re
import sys
import time
import urllib.request
from collections import defaultdict
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
sys.path.insert(0, str(REPO))
from memdiag.extract import RuleExtractor, is_complaint  # noqa: E402  (memdiag's rule layer)

MODEL = os.environ.get("MEMDIAG_MODEL", "qwen2.5:7b")
OLLAMA = "http://localhost:11434/v1/chat/completions"
LOG = []

def chat(role, prompt, as_json=False):
    body = {"model": MODEL, "temperature": 0, "seed": 0, "messages": [{"role": "user", "content": prompt}]}
    if as_json:
        body["response_format"] = {"type": "json_object"}
    req = urllib.request.Request(OLLAMA, json.dumps(body).encode(), {"Content-Type": "application/json"})
    t = time.time()
    with urllib.request.urlopen(req, timeout=600) as r:
        out = json.load(r)["choices"][0]["message"]["content"]
    LOG.append({"role": role, "secs": round(time.time() - t, 1), "output": out[:300]})
    return out

def yes(text):
    return text.strip().lower().lstrip("*\"' ").startswith("yes")

def as_obj(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        return json.loads(m.group(0)) if m else {}

print("model:", MODEL)

model: qwen2.5:7b


## Step 0. The raw agent logs
This is everything memdiag gets: a flat list of events, as exported from LangSmith, OTel, or an app's
database. Calls of type `llm` carry the memory context the agent injected. Earlier sessions are just
user and assistant messages.

In [2]:
LOGS = [
  # earlier sessions: plain chat
  {"ts": "2025-01-05T10:00", "session": "s1", "role": "user",      "text": "We just moved to Denver with my daughter Maya."},
  {"ts": "2025-01-05T10:00", "session": "s1", "role": "assistant", "text": "Welcome to Denver! How can I help you settle in?"},
  {"ts": "2025-02-10T09:00", "session": "s2", "role": "user",      "text": "Maya turns 7 next month and she's obsessed with dinosaurs."},
  {"ts": "2025-02-10T09:00", "session": "s2", "role": "assistant", "text": "Seven is a great age for dinosaurs!"},
  {"ts": "2025-03-01T18:00", "session": "s3", "role": "user",      "text": "Update: we relocated to Austin for my new job."},
  {"ts": "2025-03-01T18:00", "session": "s3", "role": "assistant", "text": "Congrats on the new job!"},
  # later calls: the agent's memory layer injected context
  {"ts": "2025-03-05T08:00", "session": "s4", "type": "llm", "query": "Recommend a weekend hike near me.",
   "context": [{"id": "m1", "text": "User lives in Denver."}],
   "response": "Try Mount Falcon Park, about 30 minutes from Denver."},
  {"ts": "2025-03-05T08:01", "session": "s4", "role": "user", "text": "We moved to Austin, remember?"},
  {"ts": "2025-03-06T12:00", "session": "s5", "type": "llm", "query": "Any gift ideas for Maya's birthday?",
   "context": [{"id": "m2", "text": "User's daughter Maya is turning 7 and loves dinosaurs."}],
   "response": "Happy to help! How old is Maya, and what is she into?"},
  {"ts": "2025-03-06T12:01", "session": "s5", "role": "user", "text": "I already told you!"},
  {"ts": "2025-03-07T15:00", "session": "s6", "type": "llm", "query": "What's my daughter's name again?",
   "context": [{"id": "m2", "text": "User's daughter Maya is turning 7 and loves dinosaurs."}],
   "response": "Your daughter's name is Maya."},
  {"ts": "2025-03-08T19:00", "session": "s7", "type": "llm", "query": "Where can I get good BBQ in town?",
   "context": [{"id": "m3", "text": "User lives in Austin."}, {"id": "m1", "text": "User lives in Denver."},
               {"id": "m4", "text": "User likes jazz."}, {"id": "m5", "text": "User is allergic to peanuts."}],
   "response": "Franklin Barbecue in Austin is a classic."},
]
# Hidden gold labels, used only in step 8 (in real logs these don't exist)
GOLD = {"s4": ("wrong", "retrieve"), "s5": ("wrong", "use"), "s6": ("right", None), "s7": ("right", None)}
print(len(LOGS), "log events")

12 log events


## Step 1. Normalize into call records
One record per LLM call: `(query, context, response)`, plus the user's next message (for symptoms).
Every user message, whether in plain chat or as a query, also goes into the user's history.

In [3]:
calls, history = [], []
for i, e in enumerate(LOGS):
    if e.get("type") == "llm":
        nxt = next((x for x in LOGS[i + 1:] if x.get("session") == e["session"] and x.get("role") == "user"), None)
        calls.append({**e, "next_user": nxt["text"] if nxt else None})
        history.append({"ts": e["ts"], "session": e["session"], "text": e["query"]})
    elif e.get("role") == "user":
        history.append({"ts": e["ts"], "session": e["session"], "text": e["text"]})

for c in calls:
    print(f"{c['session']}: Q={c['query']!r}  context={[m['id'] for m in c['context']]}  next_user={c['next_user']!r}")
print(len(history), "user messages in history")

s4: Q='Recommend a weekend hike near me.'  context=['m1']  next_user='We moved to Austin, remember?'
s5: Q="Any gift ideas for Maya's birthday?"  context=['m2']  next_user='I already told you!'
s6: Q="What's my daughter's name again?"  context=['m2']  next_user=None
s7: Q='Where can I get good BBQ in town?'  context=['m3', 'm1', 'm4', 'm5']  next_user=None
9 user messages in history


## Step 2. Symptom checks with rules only (no history, no LLM)
Cheap and runs on any log. But it misses things, as the output shows: "We moved to Austin, remember?"
is a clear complaint that the current rules don't catch. That's why the LLM steps exist.

In [4]:
for c in calls:
    c["symptom"] = bool(c["next_user"] and is_complaint(c["next_user"]))
    print(f"{c['session']}: user complaint after response? {c['symptom']}   ({c['next_user']!r})")
print("\nRule extractor on the history:", [RuleExtractor().user_facts(h["text"]) for h in history])

s4: user complaint after response? False   ('We moved to Austin, remember?')
s5: user complaint after response? True   ('I already told you!')
s6: user complaint after response? False   (None)
s7: user complaint after response? False   (None)

Rule extractor on the history: [[('city', 'Denver')], [], [], [], [('city', 'Austin')], [], [], [], []]


## Step 3. Extract structured facts from the user's own messages (LLM `extractor`)
Each fact is `(subject, attribute, value)` plus the **exact quote** it came from. Assistant messages are
ignored: the reference is what the user said, not what the agent claimed.

In [5]:
EXTRACT = (
    "Extract facts the USER states about themselves or their family. Return JSON "
    '{"facts": [{"subject": "user|<person name>", "attribute": "<snake_case, e.g. city, age, interest, '
    'child_name>", "value": "<value>", "quote": "<exact words copied from the message>"}]}. '
    "Return an empty list if there are none.\n\nMessage: ")

raw_facts = []
for h in history:
    for f in as_obj(chat("extractor", EXTRACT + h["text"], as_json=True)).get("facts", []) or []:
        if isinstance(f, dict) and f.get("attribute") and f.get("value"):
            raw_facts.append({**f, "ts": h["ts"], "session": h["session"], "source": h["text"]})
for f in raw_facts:
    print(f"{f['session']} {f['subject']}.{f['attribute']} = {f['value']!r}   quote={f.get('quote')!r}")

s1 user|myself.city = 'Denver'   quote='We just moved to Denver'
s1 user|child_name.name = 'Maya'   quote='with my daughter Maya'
s2 user|Maya.age = '7'   quote='Maya turns 7 next month'
s2 user|Maya.interest = 'dinosaurs'   quote="she's obsessed with dinosaurs"
s3 user.city = 'Austin'   quote='we relocated to Austin'
s4 user|user.city = 'Austin'   quote='We moved to Austin, remember?'


## Step 4. Grounding check (no LLM)
Keep a fact only if its quote is literally in the source message. A fact the extractor invented, or
paraphrased beyond recognition, is dropped. This is free and catches extractor hallucinations.

In [6]:
def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()

facts = [f for f in raw_facts if f.get("quote") and norm(f["quote"]) in norm(f["source"])]
dropped = [f for f in raw_facts if f not in facts]
print(f"kept {len(facts)}/{len(raw_facts)} facts; dropped:", [(f['attribute'], f['value'], f.get('quote')) for f in dropped])

kept 6/6 facts; dropped: []


## Step 5. Resolve the latest value of each fact: this is the reference
Group by `(subject, attribute)`; the latest message wins, and earlier values are kept as history.
This is what makes a stale context detectable.

In [7]:
by_key = defaultdict(list)
for f in sorted(facts, key=lambda f: f["ts"]):
    by_key[(norm(f["subject"]), norm(f["attribute"]))].append(f)

def reference_at(ts):
    """Latest value of every fact, using only messages before time ts (no peeking at the future)."""
    out = {}
    for key, fs in by_key.items():
        before = [f for f in fs if f["ts"] < ts]
        if before:
            out[key] = {"value": before[-1]["value"], "since": before[-1]["ts"],
                        "older": [f["value"] for f in before[:-1] if f["value"] != before[-1]["value"]]}
    return out

for key, r in reference_at("9999").items():
    print(f"{key[0]}.{key[1]:12s} = {r['value']!r:12s} since {r['since'][:10]}  (earlier: {r['older']})")

usermyself.city         = 'Denver'     since 2025-01-05  (earlier: [])
userchildname.name         = 'Maya'       since 2025-01-05  (earlier: [])
usermaya.age          = '7'          since 2025-02-10  (earlier: [])
usermaya.interest     = 'dinosaurs'  since 2025-02-10  (earlier: [])
user.city         = 'Austin'     since 2025-03-01  (earlier: [])
useruser.city         = 'Austin'     since 2025-03-05  (earlier: [])


## Step 6. Check each call's context on its own (no gold, no history needed)
- **Relevance** (LLM `judge_relevance`): is each injected memory relevant to the query? This gives context precision.
- **Conflicts**: does the context state two different values for the same fact? (the `extractor`, run on the memories)

In [8]:
for c in calls:
    rel = []
    for m in c["context"]:
        rel.append(yes(chat("judge_relevance",
            f"Query: {c['query']}\nMemory: {m['text']}\nIs this memory relevant to answering the query? "
            "Answer yes or no.")))
    c["relevant"] = rel
    vals = defaultdict(set)
    for m in c["context"]:
        for f in as_obj(chat("extractor", EXTRACT + m["text"], as_json=True)).get("facts", []) or []:
            if isinstance(f, dict) and f.get("attribute"):
                vals[norm(f["attribute"])].add(norm(f.get("value")))
    c["conflicts"] = {a: sorted(v) for a, v in vals.items() if len(v) > 1}
    print(f"{c['session']}: relevant {sum(rel)}/{len(rel)} {rel}   conflicts: {c['conflicts'] or 'none'}")

s4: relevant 1/1 [True]   conflicts: none


s5: relevant 1/1 [True]   conflicts: none


s6: relevant 1/1 [True]   conflicts: none


s7: relevant 2/4 [True, True, False, False]   conflicts: {'city': ['austin', 'denver']}


## Step 7. Judge sufficiency and use, then locate the failure
For each call, the judge sees the reference **as of that call's time**:
- **Sufficiency** (`judge_sufficiency`): does the context contain what the query needs, consistent with
  the user's latest facts?
- **Use** (`judge_use`): did the response use the context correctly?

Location: not sufficient -> **retrieve** (memory didn't supply it); sufficient but not used -> **use** (the
model ignored it). The judge must quote evidence; a quote that isn't in the context is flagged.

In this toy, the judge sees the whole reference (it's 5 facts). At scale you'd pass only the facts
relevant to the query.

In [9]:
def ctx_text(c):
    return "\n".join(f"- {m['text']}" for m in c["context"])

for c in calls:
    ref = reference_at(c["ts"])
    ref_txt = "\n".join(f"- {k[0]}.{k[1]} = {r['value']} (since {r['since'][:10]}"
                        + (f"; earlier {', '.join(r['older'])}" if r["older"] else "") + ")" for k, r in ref.items())
    suff = chat("judge_sufficiency",
        f"User's latest known facts:\n{ref_txt}\n\nMemory context given to the assistant:\n{ctx_text(c)}\n\n"
        f"Query: {c['query']}\n\nDoes the context contain the information this query needs, and is it "
        "consistent with the user's LATEST facts (not an outdated value)? Answer yes or no, then quote the "
        "line from the context you relied on, in double quotes.")
    used = chat("judge_use",
        f"Memory context:\n{ctx_text(c)}\n\nQuery: {c['query']}\nResponse: {c['response']}\n\n"
        "Did the response use the relevant information from the context, without ignoring or contradicting "
        "it? Answer yes or no, then one sentence why.")
    s, u = yes(suff), yes(used)
    quotes = re.findall(r'"([^"]+)"', suff)
    c["memdiag"] = {"sufficient": s, "used": u, "pass": s and u,
                    "location": None if (s and u) else ("retrieve" if not s else "use"),
                    "quote_found": any(norm(q) in norm(ctx_text(c)) for q in quotes) if quotes else None}
    m = c["memdiag"]
    print(f"{c['session']}: sufficient={s} used={u} -> {'PASS' if m['pass'] else 'FAIL at ' + m['location']}"
          f"  (judge quote found in context: {m['quote_found']})")
    print("    why (sufficiency):", suff.replace("\n", " ")[:200])
    print("    why (use):        ", used.replace("\n", " ")[:200])

s4: sufficient=False used=True -> FAIL at retrieve  (judge quote found in context: False)
    why (sufficiency): No. "The user lives in Denver."   This statement is based on the latest fact provided: "usermyself.city = Denver (since 2025-01-05)". However, there's a discrepancy as the query mentions "near me" and
    why (use):         Yes. The response used the relevant information from the context by considering that the user lives in Denver and recommended a park (Mount Falcon Park) that is near their location.


s5: sufficient=True used=True -> PASS  (judge quote found in context: True)
    why (sufficiency): Yes. "User's daughter Maya is turning 7 and loves dinosaurs."
    why (use):         Yes. The response uses the relevant information that Maya loves dinosaurs by suggesting gift ideas related to this interest.


s6: sufficient=True used=True -> PASS  (judge quote found in context: False)
    why (sufficiency): Yes. "userchildname.name = Maya (since 2025-01-05)"
    why (use):         Yes. The response used the relevant information from the context by mentioning your daughter's name, Maya.


s7: sufficient=False used=False -> FAIL at retrieve  (judge quote found in context: False)
    why (sufficiency): No.  "The context does not contain any specific information about good BBQ places in either Austin or Denver."
    why (use):         No. The response did not use any of the relevant information from the context provided about the user's location or preferences. It simply provided a recommendation for good BBQ in Austin, which is on


## Step 8. Only now: compare with the hidden gold labels

In [10]:
print("call | gold        | memdiag            | agree?")
agree_detect = agree_loc = n_wrong = 0
for c in calls:
    g_ok, g_loc = GOLD[c["session"]]
    m = c["memdiag"]
    d_ok = (g_ok == "right") == m["pass"]
    agree_detect += d_ok
    if g_ok == "wrong":
        n_wrong += 1
        agree_loc += g_loc == m["location"]
    print(f"{c['session']}   | {g_ok:5s} {str(g_loc):8s}| {'pass' if m['pass'] else 'fail'} {str(m['location']):8s}     | "
          f"detect {d_ok}, location {g_loc == m['location'] if g_ok == 'wrong' else '-'}")
print(f"\ndetection agreement {agree_detect}/{len(calls)}, location agreement {agree_loc}/{n_wrong}")

call | gold        | memdiag            | agree?
s4   | wrong retrieve| fail retrieve     | detect True, location True
s5   | wrong use     | pass None         | detect False, location False
s6   | right None    | pass None         | detect True, location -
s7   | right None    | fail retrieve     | detect False, location -

detection agreement 2/4, location agreement 1/2


## Step 9. Scores with denominators (what the dashboard would show)
Every score says what it was out of. A score with no opportunities would be `n/a`, never 1.0.

In [11]:
def score(k, n):
    return f"{k}/{n} = {k / n:.2f}" if n else "n/a (0 opportunities)"

suff_calls = [c for c in calls if c["memdiag"]["sufficient"]]
n_mem = sum(len(c["relevant"]) for c in calls)
print("context sufficiency :", score(len(suff_calls), len(calls)), "  (calls whose context held what the query needed)")
print("context use         :", score(sum(c["memdiag"]["used"] for c in suff_calls), len(suff_calls)),
      "  (of those, calls where the response used it)")
print("context precision   :", score(sum(sum(c["relevant"]) for c in calls), n_mem), "  (injected memories relevant to their query)")
print("context conflicts   :", score(sum(bool(c["conflicts"]) for c in calls), len(calls)), "  (calls with contradictory memories)")
print("user complaints     :", score(sum(c["symptom"] for c in calls), len(calls)), "  (rule-based, step 2)")
locs = [c["memdiag"]["location"] for c in calls if c["memdiag"]["location"]]
print("failures by stage   :", {loc: locs.count(loc) for loc in set(locs)} or "none")

print("\nLLM calls by role:")
by = defaultdict(lambda: [0, 0.0])
for e in LOG:
    by[e["role"]][0] += 1
    by[e["role"]][1] += e["secs"]
for role, (k, s) in by.items():
    print(f"  {role:18s} {k:3d} calls {s:6.1f}s")

context sufficiency : 2/4 = 0.50   (calls whose context held what the query needed)
context use         : 2/2 = 1.00   (of those, calls where the response used it)
context precision   : 5/7 = 0.71   (injected memories relevant to their query)
context conflicts   : 1/4 = 0.25   (calls with contradictory memories)
user complaints     : 1/4 = 0.25   (rule-based, step 2)
failures by stage   : {'retrieve': 2}

LLM calls by role:
  extractor           16 calls  230.4s
  judge_relevance      7 calls   64.5s
  judge_sufficiency    4 calls  137.0s
  judge_use            4 calls   54.7s
